# 40. 준지도 학습 — unlabeled(정상 생산 수천 샷 + 일별 불량 수) 활용

| 방법 | 내용 | unlabeled 사용 |
|---|---|---|
| M0 supervised | 기준(대조군). labeled만 RF | X |
| M1 self_training | labeled 학습 → unlabeled에서 확실한 정상·불량을 의사 라벨로 추가, 3회 반복 | 의사 라벨 |
| M2 label_spreading | labeled+unlabeled kNN 그래프에서 라벨 전파 | 그래프 |
| M3 pretrain_dae | unlabeled로 잡음 제거 오토인코더 사전학습 → 표현 8차원+복원오차를 변수로 추가 (VIME·SCARF 계열) | 표현 학습 |
| **M4 weak_daily** | **labeled 라벨 없이** unlabeled의 날짜별 불량 수(ERR_FACT_QTY)만으로 샷 불량 확률 학습 (다중 인스턴스·비율 학습) | 약한 라벨 |

**신뢰성 체크**
- 평가는 labeled를 Run 하나씩 빼고 검증. unlabeled는 labeled 시간대 ±60분을 뺀 정상 기준 데이터(20a 학습 데이터)라 평가 샷과 겹치지 않음
- `Δ vs M0`: 같은 시드·같은 fold에서 기준보다 나아졌는지 → 'unlabeled 덕분'인지 판단. 시드 3개 중 몇 개에서 좋아졌는지도 표시
- M4: 날짜 하나씩 빼고 예측한 일별 상관(ρ)을 먼저 봄 → 날짜 수준에서 신호가 없으면 샷 결과도 믿지 않음
- CN7 가스는 모두 R1 → M0~M3는 R1을 뺄 때 가스를 못 배움(`학습 못 한 Run`). **M4만 CN7을 정직하게 평가 가능**
- 마지막 셀: 좋아 보이는 방법은 순열검정으로 우연 여부 확인

**준비**: 20a·20b 실행. 실행 시간: 제품 2개 × 시드 3개 기준 20~30분 → 처음엔 `seeds=(0,)`로 빠르게 확인 권장

In [1]:
import pandas as pd
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 30)
import semi_sup as SS
R = SS.run(products=('cn7', 'rg3'), seeds=(0, ))      # 빠른 확인: seeds=(0,)

[cn7] labeled 평가 1974샷 {'정상': 1959, '가스': 11, '미성형': 4} | unlabeled 5,179샷 | 변수 38개
  M4 일별 검증: {'사용 날짜 수': 6, '날짜별 샷 수 중앙값': 793.5, '날짜 빼고 예측 Spearman ρ': -0.942857142857143, 'p': 0.004804664723032055, '변수 가중치 상위': {'Plasticizing_Position|std10': np.float64(1.077), 'Average_Back_Pressure|std10': np.float64(0.678), 'Barrel_Temperature_4|std10': np.float64(0.634), 'Max_Screw_RPM|std10': np.float64(0.629), 'Barrel_Temperature_5|std10': np.float64(0.55)}, '제품': 'cn7'}
  [cn7] seed 0 완료
[rg3] labeled 평가 591샷 {'정상': 566, '가스': 17, '미성형': 8} | unlabeled 4,359샷 | 변수 41개
  M4 일별 검증: {'사용 날짜 수': 4, '비고': '날짜가 5개 미만 → 학습 불가', '제품': 'rg3'}
  [rg3] seed 0 완료


## 1. M4 약한 지도 — 날짜 수준 검증 (ρ가 유의해야 샷 결과를 믿을 수 있음)

In [2]:
display(R['weak_info'])

,사용 날짜 수,날짜별 샷 수 중앙값,날짜 빼고 예측 Spearman ρ,p,변수 가중치 상위,제품,비고
0,6,793.5,-0.942857,0.004805,"{'Plasticizing_Position|std10': 1.077, 'Averag...",cn7,NaN
1,4,NaN,NaN,NaN,NaN,rg3,날짜가 5개 미만 → 학습 불가


## 2. 요약 — 제품 × 타깃 × 방법 (시드 평균·범위)
- 기준선: CN7 가스 2/11·미성형 3/4 (최종 모델), RG3 가스 8/17(가스 RF)·미성형 1~2/8
- `Δ② AP vs M0`가 + 이고 `Δ② AP>0 시드 수`가 3이면 unlabeled가 꾸준히 도움

In [3]:
for p in ('cn7', 'rg3'):
    print(f'===== {p.upper()} =====')
    display(R['summary'].query('제품 == @p').round(3))

===== CN7 =====


,제품,타깃,방법,② AP,AP_가스,AP_미성형,가스 평균,가스 최소,가스 최대,미성형 평균,미성형 최소,미성형 최대,Δ② AP vs M0 (평균),Δ② AP>0 시드 수,학습 못 한 Run
0,cn7,가스,M0_supervised,0.007,0.006,0.002,0.0,0,0,0.0,0,0,0.000,0,1
1,cn7,가스,M1_self_training,0.007,0.006,0.002,0.0,0,0,0.0,0,0,-0.000,0,1
2,cn7,가스,M2_label_spreading,0.007,0.006,0.002,0.0,0,0,0.0,0,0,-0.000,0,1
3,cn7,가스,M3_pretrain_dae,0.007,0.006,0.003,0.0,0,0,1.0,1,1,0.000,1,1
4,cn7,가스,M4_weak_daily,0.011,0.011,0.002,1.0,1,1,1.0,1,1,0.004,1,0
5,cn7,미성형,M0_supervised,0.007,0.006,0.002,0.0,0,0,0.0,0,0,0.000,0,1
6,cn7,미성형,M1_self_training,0.007,0.006,0.002,0.0,0,0,0.0,0,0,0.000,0,1
7,cn7,미성형,M2_label_spreading,0.007,0.006,0.002,0.0,0,0,0.0,0,0,0.000,0,1
8,cn7,미성형,M3_pretrain_dae,0.007,0.006,0.002,0.0,0,0,0.0,0,0,0.000,0,1
9,cn7,미성형,M4_weak_daily,0.011,0.011,0.002,1.0,1,1,1.0,1,1,0.004,1,0


===== RG3 =====


,제품,타깃,방법,② AP,AP_가스,AP_미성형,가스 평균,가스 최소,가스 최대,미성형 평균,미성형 최소,미성형 최대,Δ② AP vs M0 (평균),Δ② AP>0 시드 수,학습 못 한 Run
15,rg3,가스,M0_supervised,0.097,0.102,0.021,2.0,2,2,1.0,1,1,0.000,0,0
16,rg3,가스,M1_self_training,0.050,0.033,0.025,1.0,1,1,1.0,1,1,-0.047,0,0
17,rg3,가스,M2_label_spreading,0.047,0.042,0.012,4.0,4,4,0.0,0,0,-0.049,0,0
18,rg3,가스,M3_pretrain_dae,0.054,0.041,0.021,2.0,2,2,1.0,1,1,-0.043,0,0
19,rg3,미성형,M0_supervised,0.079,0.069,0.025,3.0,3,3,1.0,1,1,0.000,0,0
20,rg3,미성형,M1_self_training,0.076,0.064,0.030,2.0,2,2,2.0,2,2,-0.002,0,0
21,rg3,미성형,M2_label_spreading,0.036,0.028,0.011,1.0,1,1,0.0,0,0,-0.042,0,0
22,rg3,미성형,M3_pretrain_dae,0.063,0.056,0.018,2.0,2,2,1.0,1,1,-0.016,0,0
23,rg3,불량전체,M0_supervised,0.064,0.054,0.021,4.0,4,4,0.0,0,0,0.000,0,0
24,rg3,불량전체,M1_self_training,0.071,0.060,0.029,2.0,2,2,1.0,1,1,0.007,1,0


## 3. 시드별 상세

In [4]:
display(R['by_seed'].round(3))

,제품,방법,타깃,seed,② AP,AP_가스,AP_미성형,상위10% 가스,상위10% 미성형,Run별10% 가스,Run별10% 미성형,학습 못 한 Run 수,Δ② AP vs M0,ΔAP_가스 vs M0,ΔAP_미성형 vs M0,Δ상위10% 가스 vs M0,Δ상위10% 미성형 vs M0
0,cn7,M0_supervised,가스,0,0.007,0.006,0.002,0,0,0,0,1,0.000,0.000,0.000,0,0
1,cn7,M1_self_training,가스,0,0.007,0.006,0.002,0,0,0,0,1,-0.000,0.000,-0.000,0,0
2,cn7,M2_label_spreading,가스,0,0.007,0.006,0.002,0,0,0,0,1,-0.000,0.000,-0.001,0,0
3,cn7,M3_pretrain_dae,가스,0,0.007,0.006,0.003,0,1,0,0,1,0.000,0.000,0.000,0,1
4,cn7,M4_weak_daily,가스,0,0.011,0.011,0.002,1,1,1,1,0,0.004,0.005,0.000,1,1
5,cn7,M0_supervised,미성형,0,0.007,0.006,0.002,0,0,0,0,1,0.000,0.000,0.000,0,0
6,cn7,M1_self_training,미성형,0,0.007,0.006,0.002,0,0,0,0,1,0.000,0.000,0.000,0,0
7,cn7,M2_label_spreading,미성형,0,0.007,0.006,0.002,0,0,0,0,1,0.000,0.000,0.000,0,0
8,cn7,M3_pretrain_dae,미성형,0,0.007,0.006,0.002,0,0,0,0,1,0.000,0.000,0.000,0,0
9,cn7,M4_weak_daily,미성형,0,0.011,0.011,0.002,1,1,1,1,0,0.004,0.005,0.001,1,1


## 4. (선택) 순열검정 — 좋아 보이는 방법만
라벨 위치를 Run 안에서 무작위로 옮기고 같은 절차를 반복 → 관측 포착 수가 우연을 넘는지 (n=50이면 수 분~수십 분)

In [5]:
# res = SS.permutation_test('rg3', 'M1_self_training', '미성형', n=50)
# print(res)
# res = SS.permutation_test('cn7', 'M4_weak_daily', '가스', n=200)   # M4는 재학습이 없어 빠름
# print(res)